<a href="https://colab.research.google.com/github/divyanshuraj25/Day_20_AI_Knowledge_Assistant/blob/main/Day_20_AI_Knowledge_Assistant_FastAPI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DAY 20 — Build a Working AI Knowledge Assistant

## Objective
Build an end-to-end RAG-based AI Knowledge Assistant using:

- Python
- LangChain
- FAISS
- FastAPI
- Gemini API
- Metadata Filtering
- Grounded Prompting
- Source Citations
- Confidence Indicator

## Pipeline

User Query
    ↓
FastAPI /ask
    ↓
Metadata Filtering
    ↓
FAISS Semantic Search
    ↓
Relevant Context
    ↓
Grounded Prompt
    ↓
Gemini LLM
    ↓
Answer + Sources + Confidence

In [1]:
print("=== DAY 20 AI KNOWLEDGE ASSISTANT ===")
print("Environment setup started...")
print("Google Colab ready!")

=== DAY 20 AI KNOWLEDGE ASSISTANT ===
Environment setup started...
Google Colab ready!


In [2]:
!pip install -q fastapi uvicorn nest-asyncio pyngrok
!pip install -q faiss-cpu sentence-transformers
!pip install -q google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 71.2 MB/s eta 0:00:00


In [3]:
import fastapi
import faiss
import sentence_transformers
from google import genai

print("FastAPI:", fastapi.__version__)
print("FAISS: Installed")
print("Sentence Transformers: Installed")
print("Google GenAI: Installed")
print("✅ All packages ready!")

FastAPI: 0.141.1
FAISS: Installed
Sentence Transformers: Installed
Google GenAI: Installed
✅ All packages ready!


In [4]:
import os
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found!")

client = genai.Client(api_key=GEMINI_API_KEY)

print("✅ Gemini API connected successfully!")

✅ Gemini API connected successfully!


In [5]:
response = client.models.generate_content(
    model="gemini-2.5-flash",
    contents="Say only: Day 20 Gemini connection successful"
)

print(response.text)

ClientError: 404 NOT_FOUND. {'error': {'code': 404, 'message': 'This model models/gemini-2.5-flash is no longer available to new users. Please update your code to use models/gemini-3.8-flash for the latest features and improvements. We recommend you to use the Interactions API (https://ai.google.dev/gemini-api/docs/get-started).', 'status': 'NOT_FOUND'}}

In [7]:
MODEL_NAME = "gemini-3.8-flash"

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Day 20 Gemini connection successful"
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [8]:
models = client.models.list()

for model in models:
    if "generateContent" in (model.supported_actions or []):
        print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/lyria-3.5
models/gemini-3.1-flash-tts-preview
models/

In [9]:
MODEL_NAME = "gemini-3.1-flash-lite"

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Day 20 Gemini connection successful"
)

print(response.text)

Day 20 Gemini connection successful


In [10]:
# STEP 4 — Knowledge Base

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("✅ Embedding model loaded!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model loaded!


In [11]:
documents = [
    {
        "text": "Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.",
        "source": "ai_handbook.pdf",
        "category": "AI"
    },
    {
        "text": "Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data.",
        "source": "ml_handbook.pdf",
        "category": "ML"
    },
    {
        "text": "Deep Learning uses neural networks with multiple layers to learn complex patterns from large datasets.",
        "source": "deep_learning.pdf",
        "category": "AI"
    },
    {
        "text": "Natural Language Processing allows computers to understand, process and generate human language.",
        "source": "nlp_handbook.pdf",
        "category": "NLP"
    },
    {
        "text": "Retrieval Augmented Generation retrieves relevant information from a knowledge base before generating an answer.",
        "source": "rag_handbook.pdf",
        "category": "RAG"
    }
]

texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")

print("Embedding shape:", embeddings.shape)

Embedding shape: (5, 384)


In [12]:
def semantic_search(query, top_k=3):
    # Convert query into embedding
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True
    ).astype("float32")

    # Search FAISS
    distances, indices = index.search(query_embedding, top_k)

    results = []

    for distance, idx in zip(distances[0], indices[0]):
        if idx < len(documents):
            results.append({
                "text": documents[idx]["text"],
                "source": documents[idx]["source"],
                "category": documents[idx]["category"],
                "distance": float(distance)
            })

    return results


# Test search
query = "What is artificial intelligence?"

results = semantic_search(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Category:", result["category"])
    print("Distance:", result["distance"])
    print("Text:", result["text"])

NameError: name 'index' is not defined

In [13]:
# Recreate embeddings
texts = [doc["text"] for doc in documents]

embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True
).astype("float32")

# Create FAISS index
dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("✅ FAISS index created!")
print("Embedding shape:", embeddings.shape)
print("Documents indexed:", index.ntotal)

✅ FAISS index created!
Embedding shape: (5, 384)
Documents indexed: 5


In [14]:
query = "What is artificial intelligence?"

results = semantic_search(query, top_k=3)

for i, result in enumerate(results, 1):
    print(f"\n--- Result {i} ---")
    print("Source:", result["source"])
    print("Category:", result["category"])
    print("Distance:", result["distance"])
    print("Text:", result["text"])


--- Result 1 ---
Source: ai_handbook.pdf
Category: AI
Distance: 0.43770045042037964
Text: Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.

--- Result 2 ---
Source: ml_handbook.pdf
Category: ML
Distance: 0.8139795064926147
Text: Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data.

--- Result 3 ---
Source: nlp_handbook.pdf
Category: NLP
Distance: 1.1716837882995605
Text: Natural Language Processing allows computers to understand, process and generate human language.


In [15]:
def metadata_filtered_search(query, category=None, top_k=3):
    # First perform semantic search
    results = semantic_search(query, top_k=len(documents))

    # Apply metadata filter
    if category:
        results = [
            result for result in results
            if result["category"].lower() == category.lower()
        ]

    # Keep only top_k results
    return results[:top_k]


# Test metadata filtering
query = "What is artificial intelligence?"

filtered_results = metadata_filtered_search(
    query,
    category="AI",
    top_k=3
)

print("Query:", query)
print("Filter: category = AI")

for i, result in enumerate(filtered_results, 1):
    print(f"\n--- Filtered Result {i} ---")
    print("Source:", result["source"])
    print("Category:", result["category"])
    print("Distance:", result["distance"])
    print("Text:", result["text"])

Query: What is artificial intelligence?
Filter: category = AI

--- Filtered Result 1 ---
Source: ai_handbook.pdf
Category: AI
Distance: 0.43770045042037964
Text: Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.

--- Filtered Result 2 ---
Source: deep_learning.pdf
Category: AI
Distance: 1.3125355243682861
Text: Deep Learning uses neural networks with multiple layers to learn complex patterns from large datasets.


In [16]:
def build_grounded_prompt(query, retrieved_results):
    context = "\n\n".join(
        [
            f"Source: {result['source']}\n"
            f"Category: {result['category']}\n"
            f"Content: {result['text']}"
            for result in retrieved_results
        ]
    )

    prompt = f"""
You are an AI Knowledge Assistant.

Answer the user's question ONLY using the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the answer is not present in the context, say:
   "I don't have enough information in the provided documents."
4. Keep the answer clear and concise.
5. Mention the source document used for the answer.

CONTEXT:
{context}

USER QUESTION:
{query}

ANSWER:
"""

    return prompt


# Test the grounded prompt
query = "What is artificial intelligence?"

prompt = build_grounded_prompt(
    query,
    filtered_results
)

print(prompt)


You are an AI Knowledge Assistant.

Answer the user's question ONLY using the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the answer is not present in the context, say:
   "I don't have enough information in the provided documents."
4. Keep the answer clear and concise.
5. Mention the source document used for the answer.

CONTEXT:
Source: ai_handbook.pdf
Category: AI
Content: Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.

Source: deep_learning.pdf
Category: AI
Content: Deep Learning uses neural networks with multiple layers to learn complex patterns from large datasets.

USER QUESTION:
What is artificial intelligence?

ANSWER:



In [17]:
def generate_grounded_answer(query, retrieved_results):
    # Build grounded prompt
    prompt = build_grounded_prompt(
        query,
        retrieved_results
    )

    # Send prompt to Gemini
    response = client.models.generate_content(
        model=MODEL_NAME,
        contents=prompt
    )

    return response.text


# Test the complete RAG flow
query = "What is artificial intelligence?"

answer = generate_grounded_answer(
    query,
    filtered_results
)

print("=== AI ANSWER ===")
print(answer)

=== AI ANSWER ===
Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.

Source: ai_handbook.pdf


In [18]:
def calculate_confidence(results):
    if not results:
        return 0.0

    best_distance = results[0]["distance"]

    # Convert distance into a simple confidence score
    confidence = 1 / (1 + best_distance)

    return round(confidence * 100, 2)


def build_response(query, answer, results):
    confidence = calculate_confidence(results)

    sources = []

    for result in results:
        sources.append({
            "source": result["source"],
            "category": result["category"],
            "distance": round(result["distance"], 4)
        })

    return {
        "question": query,
        "answer": answer,
        "confidence": confidence,
        "sources": sources
    }


# Test complete response
final_response = build_response(
    query,
    answer,
    filtered_results
)

print("=== FINAL RAG RESPONSE ===")
print(final_response)

=== FINAL RAG RESPONSE ===
{'question': 'What is artificial intelligence?', 'answer': 'Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.\n\nSource: ai_handbook.pdf', 'confidence': 69.56, 'sources': [{'source': 'ai_handbook.pdf', 'category': 'AI', 'distance': 0.4377}, {'source': 'deep_learning.pdf', 'category': 'AI', 'distance': 1.3125}]}


In [19]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(
    title="AI Knowledge Assistant",
    description="RAG-based AI Knowledge Assistant",
    version="1.0"
)


class AskRequest(BaseModel):
    question: str
    category: Optional[str] = None


@app.post("/ask")
def ask_question(request: AskRequest):

    # Retrieve relevant documents
    results = metadata_filtered_search(
        request.question,
        category=request.category,
        top_k=3
    )

    # No relevant documents
    if not results:
        return {
            "question": request.question,
            "answer": "I don't have enough information in the provided documents.",
            "confidence": 0,
            "sources": []
        }

    # Generate grounded answer
    answer = generate_grounded_answer(
        request.question,
        results
    )

    # Build final response
    response = build_response(
        request.question,
        answer,
        results
    )

    return response


print("✅ FastAPI application created!")
print("Endpoint: POST /ask")

✅ FastAPI application created!
Endpoint: POST /ask


In [20]:
for route in app.routes:
    print(route.path, route.methods)

/openapi.json {'HEAD', 'GET'}
/docs {'HEAD', 'GET'}
/docs/oauth2-redirect {'HEAD', 'GET'}
/redoc {'HEAD', 'GET'}
/ask {'POST'}


In [21]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("🚀 FastAPI server started!")
print("Running on port 8000")

🚀 FastAPI server started!
Running on port 8000


In [22]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print("Status Code:", response.status_code)

INFO:     127.0.0.1:49570 - "GET /docs HTTP/1.1" 200 OK
Status Code: 200


In [23]:
import requests
import json

payload = {
    "question": "What is artificial intelligence?",
    "category": "AI"
}

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json=payload
)

print("Status Code:", response.status_code)

print("\n=== API RESPONSE ===")
print(json.dumps(response.json(), indent=2))

INFO:     127.0.0.1:50168 - "POST /ask HTTP/1.1" 500 Internal Server Error


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 422, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        self.scope, self.receive, self.send
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    )
    ^
  File "/usr/local/lib/python3.13/dist-packages/uvicorn/middleware/proxy_headers.py", line 63, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/fastapi/applications.py", line 1163, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/applications.py", line 96, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 186, i

Status Code: 500

=== API RESPONSE ===


JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [24]:
MODEL_NAME = "gemini-3.1-flash-lite"

print("Using model:", MODEL_NAME)

Using model: gemini-3.1-flash-lite


In [25]:
def generate_grounded_answer(query, retrieved_results):
    prompt = build_grounded_prompt(
        query,
        retrieved_results
    )

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        if response.text:
            return response.text

        return "The model returned an empty response."

    except Exception as e:
        print("Gemini Error:", str(e))

        return (
            "The AI model is temporarily unavailable. "
            "Please try again later."
        )

In [26]:
@app.post("/ask")
def ask_question(request: AskRequest):

    results = metadata_filtered_search(
        request.question,
        category=request.category,
        top_k=3
    )

    if not results:
        return {
            "question": request.question,
            "answer": "I don't have enough information in the provided documents.",
            "confidence": 0,
            "sources": [],
            "status": "no_relevant_context"
        }

    answer = generate_grounded_answer(
        request.question,
        results
    )

    response = build_response(
        request.question,
        answer,
        results
    )

    response["status"] = "success"

    if "temporarily unavailable" in answer.lower():
        response["status"] = "llm_unavailable"

    return response

In [ ]:
import os
os.kill(os.getpid(), 9)

In [1]:
!pip install -q fastapi uvicorn nest-asyncio faiss-cpu sentence-transformers google-genai

In [2]:
import fastapi
import faiss
import sentence_transformers
from google import genai

print("FastAPI:", fastapi.__version__)
print("FAISS: OK")
print("Sentence Transformers: OK")
print("Google GenAI: OK")
print("✅ Fresh runtime is ready!")

FastAPI: 0.141.1
FAISS: OK
Sentence Transformers: OK
Google GenAI: OK
✅ Fresh runtime is ready!


In [3]:
from google.colab import userdata
from google import genai

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY not found in Colab Secrets")

client = genai.Client(api_key=GEMINI_API_KEY)

MODEL_NAME = "gemini-3.1-flash-lite"

print("✅ Gemini client initialized")
print("Model:", MODEL_NAME)

✅ Gemini client initialized
Model: gemini-3.1-flash-lite


In [4]:
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Day 20 Gemini is working"
)

print(response.text)

ServerError: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}

In [5]:
MODEL_NAME = "gemini-3.5-flash-lite"

print("Testing:", MODEL_NAME)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Day 20 Gemini is working"
)

print(response.text)

Testing: gemini-3.5-flash-lite
Day 20 Gemini is working


In [6]:
MODEL_NAME = "gemini-3.5-flash-lite"

print("Testing:", MODEL_NAME)

response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Say only: Day 20 Gemini is working"
)

print(response.text)

Testing: gemini-3.5-flash-lite
Day 20 Gemini is working


In [7]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

# Knowledge base
documents = [
    {
        "text": "Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.",
        "source": "ai_handbook.pdf",
        "category": "AI"
    },
    {
        "text": "Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data.",
        "source": "ml_handbook.pdf",
        "category": "ML"
    },
    {
        "text": "Deep Learning uses neural networks with multiple layers to learn complex patterns from large datasets.",
        "source": "deep_learning.pdf",
        "category": "AI"
    },
    {
        "text": "Natural Language Processing allows computers to understand, process and generate human language.",
        "source": "nlp_handbook.pdf",
        "category": "NLP"
    },
    {
        "text": "Retrieval Augmented Generation retrieves relevant information from a knowledge base before generating an answer.",
        "source": "rag_handbook.pdf",
        "category": "RAG"
    }
]

# Extract text
texts = [doc["text"] for doc in documents]

# Create embeddings
embeddings = embedding_model.encode(
    texts,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

# FAISS cosine similarity using Inner Product
dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)

print("Embedding shape:", embeddings.shape)
print("FAISS vectors:", index.ntotal)
print("Knowledge base ready!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embedding shape: (5, 384)
FAISS vectors: 5
Knowledge base ready!


In [8]:
def semantic_search(query, top_k=3):
    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")

    similarities, indices = index.search(query_embedding, top_k)

    results = []

    for similarity, idx in zip(similarities[0], indices[0]):
        if idx < len(documents):
            results.append({
                "text": documents[idx]["text"],
                "source": documents[idx]["source"],
                "category": documents[idx]["category"],
                "similarity": float(similarity)
            })

    return results


def metadata_filtered_search(query, category=None, top_k=3):
    results = semantic_search(query, top_k=len(documents))

    if category:
        results = [
            result for result in results
            if result["category"].lower() == category.lower()
        ]

    return results[:top_k]


# Test search
results = metadata_filtered_search(
    "What is artificial intelligence?",
    category="AI",
    top_k=3
)

for i, result in enumerate(results, 1):
    print(f"\nResult {i}")
    print("Source:", result["source"])
    print("Category:", result["category"])
    print("Similarity:", round(result["similarity"], 4))
    print("Text:", result["text"])


Result 1
Source: ai_handbook.pdf
Category: AI
Similarity: 0.7811
Text: Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence.

Result 2
Source: deep_learning.pdf
Category: AI
Similarity: 0.3437
Text: Deep Learning uses neural networks with multiple layers to learn complex patterns from large datasets.


In [9]:
def build_grounded_prompt(query, retrieved_results):
    context = "\n\n".join(
        [
            f"Source: {result['source']}\n"
            f"Category: {result['category']}\n"
            f"Content: {result['text']}"
            for result in retrieved_results
        ]
    )

    prompt = f"""
You are an AI Knowledge Assistant.

Answer the user's question ONLY using the provided context.

Rules:
1. Do not use outside knowledge.
2. Do not invent facts.
3. If the answer is not present in the context, say:
   "I don't have enough information in the provided documents."
4. Keep the answer clear and concise.
5. Include source citations in this format:
   [Source: filename]

CONTEXT:
{context}

USER QUESTION:
{query}

ANSWER:
"""

    return prompt


def generate_grounded_answer(query, retrieved_results):
    prompt = build_grounded_prompt(query, retrieved_results)

    try:
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=prompt
        )

        if response.text:
            return response.text.strip()

        return "The model returned an empty response."

    except Exception as e:
        print("Gemini Error:", str(e))

        return (
            "The AI model is temporarily unavailable. "
            "Please try again later."
        )


# Test grounded answer
query = "What is artificial intelligence?"

results = metadata_filtered_search(
    query,
    category="AI",
    top_k=3
)

answer = generate_grounded_answer(query, results)

print("QUESTION:")
print(query)

print("\nANSWER:")
print(answer)

QUESTION:
What is artificial intelligence?

ANSWER:
Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence [Source: ai_handbook.pdf].


In [10]:
def build_response(query, answer, results):
    if not results:
        return {
            "query": query,
            "answer": "I don't have enough information in the provided documents.",
            "confidence": 0,
            "warning": "No relevant documents were retrieved.",
            "sources": []
        }

    # Highest similarity score
    best_similarity = max(
        result["similarity"] for result in results
    )

    confidence = round(best_similarity * 100, 2)

    response = {
        "query": query,
        "answer": answer,
        "confidence": confidence,
        "sources": [
            {
                "source": result["source"],
                "category": result["category"],
                "similarity": round(result["similarity"], 4)
            }
            for result in results
        ]
    }

    # Day 20 requirement:
    # Warn when highest similarity is below 0.3
    if best_similarity < 0.3:
        warning = (
            "Low confidence: the retrieved context may not "
            "sufficiently support this answer."
        )

        response["warning"] = warning
        response["answer"] += f"\n\n⚠️ {warning}"

    return response


# Test final response
query = "What is artificial intelligence?"

results = metadata_filtered_search(
    query,
    category="AI",
    top_k=3
)

answer = generate_grounded_answer(
    query,
    results
)

final_response = build_response(
    query,
    answer,
    results
)

import json

print(json.dumps(final_response, indent=2))

{
  "query": "What is artificial intelligence?",
  "answer": "Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence [Source: ai_handbook.pdf].",
  "confidence": 78.11,
  "sources": [
    {
      "source": "ai_handbook.pdf",
      "category": "AI",
      "similarity": 0.7811
    },
    {
      "source": "deep_learning.pdf",
      "category": "AI",
      "similarity": 0.3437
    }
  ]
}


In [11]:
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Optional

app = FastAPI(
    title="AI Knowledge Assistant",
    description="RAG-based AI Knowledge Assistant using FAISS and Gemini",
    version="1.0"
)


class AskRequest(BaseModel):
    query: str
    category: Optional[str] = None


@app.post("/ask")
def ask_question(request: AskRequest):

    # 1. Retrieve relevant documents
    results = metadata_filtered_search(
        request.query,
        category=request.category,
        top_k=3
    )

    # 2. No relevant documents
    if not results:
        return {
            "query": request.query,
            "answer": "I don't have enough information in the provided documents.",
            "confidence": 0,
            "warning": "No relevant documents were retrieved.",
            "sources": []
        }

    # 3. Generate grounded answer
    answer = generate_grounded_answer(
        request.query,
        results
    )

    # 4. Build final response
    response = build_response(
        request.query,
        answer,
        results
    )

    return response


print("FastAPI application created successfully!")

FastAPI application created successfully!


In [12]:
import nest_asyncio
import threading
import uvicorn

nest_asyncio.apply()

def run_server():
    uvicorn.run(
        app,
        host="0.0.0.0",
        port=8000,
        log_level="info"
    )

server_thread = threading.Thread(
    target=run_server,
    daemon=True
)

server_thread.start()

print("FastAPI server started!")
print("API Docs: http://127.0.0.1:8000/docs")

FastAPI server started!
API Docs: http://127.0.0.1:8000/docs


In [13]:
import requests

response = requests.get("http://127.0.0.1:8000/docs")

print("Status Code:", response.status_code)
print("Server is running:", response.status_code == 200)

INFO:     127.0.0.1:51156 - "GET /docs HTTP/1.1" 200 OK
Status Code: 200
Server is running: True


In [14]:
import requests
import json

payload = {
    "query": "What is artificial intelligence?",
    "category": "AI"
}

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json=payload
)

print("Status Code:", response.status_code)
print(json.dumps(response.json(), indent=2))

INFO:     127.0.0.1:41164 - "POST /ask HTTP/1.1" 200 OK
Status Code: 200
{
  "query": "What is artificial intelligence?",
  "answer": "Artificial Intelligence is the field of building computer systems that can perform tasks requiring human-like intelligence [Source: ai_handbook.pdf].",
  "confidence": 78.11,
  "sources": [
    {
      "source": "ai_handbook.pdf",
      "category": "AI",
      "similarity": 0.7811
    },
    {
      "source": "deep_learning.pdf",
      "category": "AI",
      "similarity": 0.3437
    }
  ]
}


In [15]:
test_queries = [
    ("What is artificial intelligence?", "AI"),
    ("What is machine learning?", "ML"),
    ("What is deep learning?", "AI"),
    ("What is natural language processing?", "NLP"),
    ("What is retrieval augmented generation?", "RAG"),

    ("How does machine learning relate to artificial intelligence?", None),
    ("How does deep learning learn patterns?", None),
    ("What does natural language processing allow computers to do?", None),
    ("How does RAG improve question answering?", None),
    ("What is the purpose of a knowledge base?", None),

    ("Explain the relationship between AI and ML.", None),
    ("Compare machine learning and deep learning.", None),
    ("How can NLP process human language?", None),
    ("Why does RAG retrieve information before generating an answer?", None),
    ("What information is available about computer vision?", None)
]

test_results = []

for i, (query, category) in enumerate(test_queries, 1):

    payload = {
        "query": query
    }

    if category:
        payload["category"] = category

    try:
        response = requests.post(
            "http://127.0.0.1:8000/ask",
            json=payload
        )

        data = response.json()

        similarities = [
            source["similarity"]
            for source in data.get("sources", [])
        ]

        best_similarity = max(similarities) if similarities else 0

        test_results.append({
            "Test": i,
            "Query": query,
            "Category": category if category else "None",
            "Retrieval Similarity": round(best_similarity, 4),
            "Confidence": data.get("confidence", 0),
            "Status": response.status_code
        })

    except Exception as e:
        test_results.append({
            "Test": i,
            "Query": query,
            "Category": category if category else "None",
            "Retrieval Similarity": 0,
            "Confidence": 0,
            "Status": "ERROR"
        })

import pandas as pd

results_df = pd.DataFrame(test_results)

display(results_df)

INFO:     127.0.0.1:47578 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47582 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47598 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47610 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:54614 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:54618 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:54620 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:54632 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:54640 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40134 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40144 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40154 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40156 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40160 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:40166 - "POST /ask HTTP/1.1" 200 OK


,Test,Query,Category,Retrieval Similarity,Confidence,Status
0,1,What is artificial intelligence?,AI,0.7811,78.11,200
1,2,What is machine learning?,ML,0.8225,82.25,200
2,3,What is deep learning?,AI,0.6298,62.98,200
3,4,What is natural language processing?,NLP,0.8118,81.18,200
4,5,What is retrieval augmented generation?,RAG,0.6750,67.50,200
5,6,How does machine learning relate to artificial...,None,0.7707,77.07,200
6,7,How does deep learning learn patterns?,None,0.7736,77.36,200
7,8,What does natural language processing allow co...,None,0.8608,86.08,200
8,9,How does RAG improve question answering?,None,0.3527,35.27,200
9,10,What is the purpose of a knowledge base?,None,0.4127,41.27,200


In [16]:
def retrieval_quality_score(similarity):
    if similarity >= 0.75:
        return 5
    elif similarity >= 0.60:
        return 4
    elif similarity >= 0.45:
        return 3
    elif similarity >= 0.30:
        return 2
    else:
        return 1


def response_quality_score(data):
    answer = data.get("answer", "").strip()

    if not answer:
        return 1

    score = 1

    # Relevant answer exists
    if "I don't have enough information" not in answer:
        score += 2

    # Source citation exists
    if "[Source:" in answer:
        score += 1

    # Retrieved sources exist
    if len(data.get("sources", [])) > 0:
        score += 1

    return min(score, 5)


quality_results = []

for i, (query, category) in enumerate(test_queries, 1):

    payload = {"query": query}

    if category:
        payload["category"] = category

    try:
        response = requests.post(
            "http://127.0.0.1:8000/ask",
            json=payload
        )

        data = response.json()

        similarities = [
            source["similarity"]
            for source in data.get("sources", [])
        ]

        best_similarity = max(similarities) if similarities else 0

        quality_results.append({
            "Test": i,
            "Query": query,
            "Retrieval Similarity": round(best_similarity, 4),
            "Retrieval Quality (1-5)": retrieval_quality_score(
                best_similarity
            ),
            "Response Quality (1-5)": response_quality_score(
                data
            ),
            "Confidence": data.get("confidence", 0)
        })

    except Exception as e:
        quality_results.append({
            "Test": i,
            "Query": query,
            "Retrieval Similarity": 0,
            "Retrieval Quality (1-5)": 1,
            "Response Quality (1-5)": 1,
            "Confidence": 0
        })


quality_df = pd.DataFrame(quality_results)

display(quality_df)

INFO:     127.0.0.1:41668 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:41676 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:41692 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47664 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47670 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47672 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47688 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47704 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:47706 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43728 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43736 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43746 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43752 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43768 - "POST /ask HTTP/1.1" 200 OK
INFO:     127.0.0.1:43784 - "POST /ask HTTP/1.1" 200 OK


,Test,Query,Retrieval Similarity,Retrieval Quality (1-5),Response Quality (1-5),Confidence
0,1,What is artificial intelligence?,0.7811,5,5,78.11
1,2,What is machine learning?,0.8225,5,5,82.25
2,3,What is deep learning?,0.6298,4,5,62.98
3,4,What is natural language processing?,0.8118,5,5,81.18
4,5,What is retrieval augmented generation?,0.6750,4,5,67.50
5,6,How does machine learning relate to artificial...,0.7707,5,5,77.07
6,7,How does deep learning learn patterns?,0.7736,5,5,77.36
7,8,What does natural language processing allow co...,0.8608,5,5,86.08
8,9,How does RAG improve question answering?,0.3527,2,2,35.27
9,10,What is the purpose of a knowledge base?,0.4127,2,2,41.27


In [17]:
query = "What is quantum entanglement?"

payload = {
    "query": query
}

response = requests.post(
    "http://127.0.0.1:8000/ask",
    json=payload
)

data = response.json()

print("Query:", query)
print("Status Code:", response.status_code)
print("Confidence:", data.get("confidence"))

print("\nAnswer:")
print(data.get("answer"))

print("\nWarning:")
print(data.get("warning", "No low-confidence warning"))

INFO:     127.0.0.1:45320 - "POST /ask HTTP/1.1" 200 OK
Query: What is quantum entanglement?
Status Code: 200
Confidence: 18.98

Answer:
I don't have enough information in the provided documents.

⚠️ Low confidence: the retrieved context may not sufficiently support this answer.

Low confidence: the retrieved context may not sufficiently support this answer.


In [18]:
print("=" * 55)
print("DAY 20 — AI KNOWLEDGE ASSISTANT")
print("=" * 55)

print("✓ Knowledge base created")
print("✓ Sentence Transformer embeddings")
print("✓ FAISS semantic search")
print("✓ Metadata filtering")
print("✓ Grounded Gemini prompting")
print("✓ Source citations")
print("✓ FastAPI POST /ask endpoint")
print("✓ 15 test queries evaluated")
print("✓ Retrieval quality scoring")
print("✓ Response quality scoring")
print("✓ Low-confidence warning (< 0.3)")
print("✓ Out-of-context question handling")

print("\nAPI Endpoint:")
print("POST /ask")

print("\nExample Request:")
print({
    "query": "What is artificial intelligence?",
    "category": "AI"
})

print("\nDay 20 Status: COMPLETE")

DAY 20 — AI KNOWLEDGE ASSISTANT
✓ Knowledge base created
✓ Sentence Transformer embeddings
✓ FAISS semantic search
✓ Metadata filtering
✓ Grounded Gemini prompting
✓ Source citations
✓ FastAPI POST /ask endpoint
✓ 15 test queries evaluated
✓ Retrieval quality scoring
✓ Response quality scoring
✓ Low-confidence warning (< 0.3)
✓ Out-of-context question handling

API Endpoint:
POST /ask

Example Request:
{'query': 'What is artificial intelligence?', 'category': 'AI'}

Day 20 Status: COMPLETE
